# Vérification du téléchargement Lidar HD

Ce notebook vérifie que `download.lidar` interroge correctement l'index des dalles Lidar HD (IGN) sur l'emprise d'une entité du CSV, et visualise les emprises de dalles trouvées.

⚠️ Chaque dalle `.copc.laz` pèse couramment plusieurs centaines de Mo. Le téléchargement réel (section 4) est **désactivé par défaut** pour ne pas déclencher plusieurs Go de téléchargement à chaque exécution de ce notebook.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities, TARGET_CRS
from download.wfs_client import fetch_wfs_features
from download.limites_administratives import fetch_commune_boundary, resolve_bbox
from download.lidar import (
    TILE_INDEX_WFS_URL,
    TILE_INDEX_TYPENAME,
    TILE_URL_FIELD,
    TILE_FILENAME_FIELD,
    download_lidar,
)

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Index des dalles Lidar HD intersectant l'emprise

In [ ]:
bbox = resolve_bbox(entity)
tiles = fetch_wfs_features(TILE_INDEX_WFS_URL, TILE_INDEX_TYPENAME, bbox, TARGET_CRS)
print("Nombre de dalles trouvees :", len(tiles))
tiles[[TILE_FILENAME_FIELD, "format", TILE_URL_FIELD]]

## 3. Visualisation des emprises de dalles

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

boundary = fetch_commune_boundary(entity.code_insee)

tiles_wgs84 = tiles.to_crs(epsg=4326)
boundary_wgs84 = boundary.to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(
    tiles_wgs84,
    layer_name="Dalles Lidar HD",
    style={"color": "purple", "fillOpacity": 0.05},
    info_mode="on_click",
)
m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m

## 4. Téléchargement réel (optionnel)

Passer `DOWNLOAD_TILES = True` pour déclencher le téléchargement effectif des dalles ci-dessus dans `data/lidar/` (plusieurs Go possibles selon le nombre de dalles).

In [ ]:
DOWNLOAD_TILES = False

if DOWNLOAD_TILES:
    written = download_lidar(bbox)
    written
else:
    print("Telechargement desactive (DOWNLOAD_TILES = False).")